### **Student Information**
GitHub ID: khanhnguyen3007 - https://github.com/khanhnguyen3007/DM2026-Lab2-Exercise

Kaggle name: nqkhanh3007 - Nguyễn Quốc Khánh

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**

## 1. Model Development (10 pts Required)

**Describe briefly each section, you can add graphs/charts to support your explanations.**

### 1.1 Preprocessing Steps

Dataset loading:
   * Locates data_identification.csv, emotion.csv, and final_posts.json under /kaggle/input
   * Normalizes `post_id` by casting to `str` and stripping whitespace (`.str.strip()`), then removes duplicate `post_id` entries across train/test IDs, emotion labels, and post texts, keeping the first occurrence
   * Creates `df_train` by inner-joining train IDs with `text` and `label`, dropping rows with missing `text` or `label`, and normalizing `label` with `.astype(str).str.strip().str.lower()`
   * Creates `df_test` by left-joining test IDs with `text`, and filling missing `text` values with an empty string `""` (`text_raw`), using a left join to ensure all test entries are preserved

Preprocessing steps:
   * Compiles regex patterns (`URL_PATTERN`, `MENTION_PATTERN`, `HASHTAG_PATTERN`, `REPEATED_CHAR_PATTERN`, `WHITESPACE_PATTERN`)
   * Converts HTML entities to characters using `html.unescape`
   * Replaces URLs (`http\S+|www\.\S+`) with a single space `" "`
   * Replaces `@username` mentions (`@\w+`) with `"@user"`
   * Removes `#` from hashtags (`#(\w+)`) while keeping the hashtag word (`\1`)
   * Reduces 3 or more repeated characters (`(.)\1{2,}`) to 2 characters (`\1\1`)
   * Collapses consecutive whitespace (`\s+`) into a single space and trims leading/trailing whitespace, preserving casing and punctuation
* Post-cleaning handling and label encoding:
  * Applies `preprocess_text` to `df_train` and removes training rows where `text_clean` has length `0`
  * Applies `preprocess_text` to `df_test` and falls back to `text_raw.str.strip()` for rows where `text_clean` is empty
  * Encodes `df_train["label"]` into integer `label_id` (`y_all`) using `LabelEncoder`, and prints the processed train/test counts, fold count, and class list


### 1.2 Feature Engineering Steps

I employ two feature engineering techniques: Transformer-based encoding and TF-IDF-based feature extraction:
   * Token encoding for Transformers:
        * Initialize the tokenizer from MODEL_NAME using AutoTokenizer.from_pretrained(MODEL_NAME), load the tokenizer corresponding to the Transformer model specified in MODEL_NAME (the `emotion-english-roberta-large` model). The tokenizer converts text into tokens that the model can understand
        * In `PreTokenizedEmotionDataset.__init__(self, texts, labels=None)`, perform tokenization of the input list of texts using `tokenizer(...)`, and the resulting data is then converted into PyTorch tensors
        * Implement the `__len__` method to return the number of tokenized sequences, and the `__getitem__` method to return a dictionary containing input_ids, attention_mask, and labels
   * Multi-level TF-IDF feature extraction:
        * Implements a `create_tfidf_vectorizer` function that combines word-level and character level TF-IDF vectorizers using `FeatureUnion`
                * Word-level TF-IDF vectorizer (`word_tfidf`): Extract word unigrams and bigrams `ngram_range=(1, 2)`, with a maximum limit of `max_features=30000` , set `min_df=1`, enable `sublinear_tf=True`, and output as `np.float32`. Using bigrams helps the model capture the context between words, rather than just considering each word in isolation
                * Character-level TF-IDF vectorizer (`char_tfidf`): Using `analyzer="char_wb"` with an n-gram range of 3 to 5 characters `ngram_range=(3, 5)`, with a maximum limit of `max_features=40000`, set `min_df=2`, enable `sublinear_tf=True`, and output as `np.float32`. Character-level features can help identify variations in writing style, abbreviations, spelling errors, or the elongation of characters within texts
        * The combined vectorizer (constructed using `FeatureUnion`) is then used to fit and transform the training data, as well as to transform the validation and test datasets, resulting in sparse matrices that contain TF-IDF features for each sample. Combining these two types of features allows for a more diverse representation of the text, which is particularly useful when dealing with social media data

### 1.3 Explanation of Your Model

**I use a combination of two models `RoBERTa and TF-IDF` to select the best result:**
   * Initialization, Model Training and Optimization:
        * The `build_transformer()` function is responsible for initializing the RoBERTa-large model and the components required for the training process; it uses an inverse class frequency weight of `0.7` (I experimented with various weights ranging from 0.4 to 1.0 and ultimately selected 0.7, as it yielded the best results). In this approach, `from_pretrained` is used to fine-tune the model for a 6-class emotion classification task, with `CrossEntropyLoss` employed to calculate the loss for multi-class classification
        * Implement the `train_epoch_amp()` function to carry out the model training process for a single epoch. Use `torch.amp.autocast()` for mixed-precision training; skip a batch if the loss value is not finite (`not torch.isfinite(raw_loss)`). Divide the loss by `GRAD_ACCUM_STEPS = 4` (given `BATCH_SIZE = 8`), perform gradient clipping using `clip_grad_norm_()`, and then update the optimizer and the learning rate scheduler. After each group of mini-batches, update the model parameters using the AdamW algorithm. Clear old gradients and update the learning rate according to the scheduler
        * Use the `predict_loader_probs` function with `torch.no_grad()` and `torch.softmax(outputs.logits.float(), dim=-1)` to obtain validation probabilities after each epoch
   * Stratified 5-Fold Training: I would like to perform a comprehensive evaluation on the training data; using an 80/20 train/validation split would leave the training data incomplete, whereas conducting multiple evaluations in this manner allows me to obtain a more holistic set of results
      *   With N_SPLITS = 5, the data is divided into 5 `StratifiedKFold` folds, ensuring that the distribution of sentiment classes in each fold closely mirrors the distribution across the entire dataset
      * In each fold, the pipeline deploys two model branches:
        * TF-IDF + Logistic Regression + Calibrated LinearSVC: TF-IDF + Logistic Regression + Calibrated LinearSVC: Word-level and character-level TF-IDF are fitted on the training fold, using Logistic Regression and Calibrated LinearSVC as the two classifiers. The two models are combined by averaging their predicted probabilities with a 50:50 weighting
        * RoBERTa-large fine-tuning: For each fold, perform the following steps: initialize the RoBERTa-large model, train it on the training fold, evaluate it on the validation fold after each epoch, and calculate the Macro F1 score to monitor performance. Following each epoch, compute the probabilities for the validation set and save the model state that achieves the best performance (based on the F1 score on the validation set) for that specific fold
      * After completing the 5 folds, we obtain the full out-of-fold (`OOF`) probability matrices (`oof_roberta`, `oof_tfidf`) for the entire training set, as well as the fold-specific probability predictions for the test set (`test_roberta_folds`, `test_tfidf_folds`)
   * Combine the predicted probabilities from the two branches: `oof_blend = alpha * oof_roberta + (1.0 - alpha) * oof_tfidf` - alpha in the range [0.60,1.00] (41 points) on OOF, used to select the optimal blending weights between oof_roberta and oof_tfidf for generating oof_best_probs and test_best_probs
   * The `optimize_class_multipliers()` function is used to determine the optimal multipliers for each class, such that the Macro F1 score on the OOF predictions is maximized. This process is carried out in several steps: initializing a set of weights at 1.0, along with 10 additional sets of random weights ranging from 0.80 to 1.30 to serve as starting points for optimization; employing a coordinate-wise search to refine the weights for each class; and subsequently applying Powell optimization method to further improve the set of weights. These final class multipliers are then applied to the blended probabilities to generate the ultimate OOF predictions, as well as the predictions for the test set.
   * For the final step, I began by calculating the average of the predicted probabilities across the 5 folds, then applied the optimized class multipliers. Next, I used `argmax` to select the class with the highest probability and `LabelEncoder.inverse_transform()` to convert the class IDs back into their original sentiment labels. Finally, I created a DataFrame with two columns`id` and `label` and exported the results to a file named `sampleSubmission.csv`

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

I have conducted experiments using baseline models (`TF-IDF + MiniLM`) combined with various classifiers (`SVC, MLP, NB, and KNN`), but the results have not been very promising, with scores ranging only from `0.46 to 0.58`. I believe this is the best performance achievable with these baseline models. I have sought assistance from AI to explore ways to optimize the approach and improve my score for the competition. The experimental results are shown in the image below:

![pic_ranking.png](./pics/exp_experimentation.png)

I asked an AI for assistance with the following prompt:
  * "I am using TF-IDF + MiniLM models combined with SVC, MLP, NB, and KNN, but the macro F1 score is not very high. Is there any way I can improve this score?"
    * The AI provided several suggestions: tuning the `C` parameter for `LinearSVC`, adjusting the `ngram_range` to `(1, 2)`, lowering `min_df` to 1 while increasing `max_features`, and combining features from both TF-IDF and MiniLM embeddings. I implemented all of these recommendations, yet there has been no positive change in the score; it appears that, given the simplicity of this model, the performance has reached its absolute ceiling
    * One solution that proved highly effective for my problem is `Stratified K-Fold`. Initially, I used an 80/20 split for the training and validation sets. However, given that the training dataset is quite small —consisting of just over 3,000 samples— this approach resulted in a significant shortage of data for training. Consequently, I applied 5-Fold cross-validation to my training dataset. This allows me to evaluate the model across the entire training set, ensuring the most accurate training results possible
  * "I would like to use a transformer for this tweet text; do you have any models you could suggest?"
    * The AI suggested several transformer models suitable for text classification tasks — for instance, `sentence-transformers/all-mpnet-base-v2`, `BAAI/bge-base-en-v1.5`, or models fine-tuned for sentiment analysis or processing tweets, available on Hugging Face (such as the `cardiffnlp/twitter-roberta-...` series or `j-hartmann/emotion-english-distilroberta-base`). After reviewing these suggestions, I experimented with the `emotion-english-distilroberta-base` model. With this initial choice, my score dropped from 0.58 to 0.13; I realized that I likely hadn't integrated the model correctly—the way it interacted with the classification layer and the existing training configuration was incompatible—so I decided to explore other options
    * Next, I experimented with the `j-hartmann/emotion-english-roberta-large` model — a larger and more powerful model. It has been fine-tuned specifically for emotion classification tasks, making it highly suitable for the problem presented in the competition I participated in. Upon applying this model, I observed a significant improvement in the Macro F1 score, reaching 0.63 (which also served as my final result in the Kaggle competition). This demonstrates that selecting an appropriate transformer model—one aligned with the specific requirements of the task—plays a pivotal role in the data mining process. Making the right choice was a key factor in enhancing performance for this problem

---

# **Competition Code**

## 1. Preprocessing Steps

In [ ]:
import gc
import html
import json
import math
import os
import random
import re
import time



import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.sparse import hstack
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_cosine_schedule_with_warmup,
)

# 1. Configuration
INPUT_PATH = "/kaggle/input"
MODEL_NAME = "j-hartmann/emotion-english-roberta-large"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
N_SPLITS = 5
MAX_LEN = 96
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 4
EPOCHS = 3
LEARNING_RATE = 1.5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
LABEL_SMOOTHING = 0.02
USE_GRAD_CHECKPOINTING = False
RANDOM_STATE = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def clear_cuda_memory(*objs):
    for obj in objs:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


seed_everything(RANDOM_STATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_cuda = torch.cuda.is_available()
amp_dtype = torch.float16 if use_cuda else torch.float32
use_grad_scaler = use_cuda and (amp_dtype == torch.float16)

print(f"Using device: {device} | AMP dtype: {amp_dtype if use_cuda else 'float32'}")

# 2. Locate and load datasets

def find_file(filename, base_dir=INPUT_PATH):
    for root, _, files in os.walk(base_dir):
        if filename in files:
            return os.path.join(root, filename)
    raise FileNotFoundError(f"File '{filename}' not found in '{base_dir}'.")


id_path = find_file("data_identification.csv")
emotion_path = find_file("emotion.csv")
posts_path = find_file("final_posts.json")

df_id = pd.read_csv(id_path, dtype={"post_id": str})
df_emotion = pd.read_csv(emotion_path, dtype={"post_id": str})

with open(posts_path, "r", encoding="utf-8") as f:
    raw_json = json.load(f)

if isinstance(raw_json, dict) and "root" in raw_json:
    records = raw_json["root"]
elif isinstance(raw_json, list):
    records = raw_json
else:
    raise ValueError("Unexpected JSON structure.")

df_posts = pd.DataFrame(records, columns=["post_id", "text"])
df_posts["post_id"] = df_posts["post_id"].astype(str)

for df in (df_id, df_emotion, df_posts):
    df["post_id"] = df["post_id"].str.strip()

df_train_id = df_id.loc[df_id["set"] == "train", ["post_id"]].drop_duplicates(subset="post_id")
df_test_id = df_id.loc[df_id["set"] == "test", ["post_id"]].drop_duplicates(subset="post_id")
df_emotion_clean = df_emotion.drop_duplicates(subset="post_id")
df_posts_clean = df_posts.drop_duplicates(subset="post_id")

df_train = (
    df_train_id
    .merge(df_posts_clean[["post_id", "text"]], on="post_id", how="inner")
    .merge(df_emotion_clean[["post_id", "label"]], on="post_id", how="inner")
    .dropna(subset=["text", "label"])
    .drop_duplicates(subset="post_id")
    .reset_index(drop=True)
)
df_train["label"] = df_train["label"].astype(str).str.strip().str.lower()



#3. Preprocessing

URL_PATTERN = re.compile(r"http\S+|www\.\S+")
MENTION_PATTERN = re.compile(r"@\w+")
HASHTAG_PATTERN = re.compile(r"#(\w+)")
REPEATED_CHAR_PATTERN = re.compile(r"(.)\1{2,}")
WHITESPACE_PATTERN = re.compile(r"\s+")


def preprocess_text(text):
    text = html.unescape(str(text))
    text = URL_PATTERN.sub(" ", text)
    text = MENTION_PATTERN.sub("@user", text)
    text = HASHTAG_PATTERN.sub(r"\1", text)
    text = REPEATED_CHAR_PATTERN.sub(r"\1\1", text)
    text = WHITESPACE_PATTERN.sub(" ", text).strip()
    return text


df_train["text_clean"] = df_train["text"].map(preprocess_text)
df_train = df_train.loc[df_train["text_clean"].str.len() > 0].reset_index(drop=True)

label_encoder = LabelEncoder()
df_train["label_id"] = label_encoder.fit_transform(df_train["label"])
num_classes = len(label_encoder.classes_)
y_all = df_train["label_id"].to_numpy()

df_test = df_test_id.merge(df_posts_clean[["post_id", "text"]], on="post_id", how="left")
df_test["text_raw"] = df_test["text"].fillna("")
df_test["text_clean"] = df_test["text_raw"].map(preprocess_text)
empty_mask = df_test["text_clean"].str.len() == 0
df_test.loc[empty_mask, "text_clean"] = df_test.loc[empty_mask, "text_raw"].str.strip()

print(f"Full train: {len(df_train)} | Test: {len(df_test)} | Folds: {N_SPLITS}")
print("Classes:", list(label_encoder.classes_))

## 2. Feature Engineering Steps

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PreTokenizedEmotionDataset(Dataset):
    def __init__(self, texts, labels=None):
        enc = tokenizer(
            list(texts),
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN
        )
        self.input_ids = torch.tensor(enc["input_ids"], dtype=torch.long)
        self.attention_mask = torch.tensor(enc["attention_mask"], dtype=torch.long)
        self.labels = torch.tensor(list(labels), dtype=torch.long) if labels is not None else None

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        item = {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
        }
        if self.labels is not None:
            item["labels"] = self.labels[idx]
        return item
def create_tfidf_vectorizer():
    return FeatureUnion([
        (
            "word_tfidf",
            TfidfVectorizer(
                ngram_range=(1, 2),
                max_features=30000,
                min_df=1,
                sublinear_tf=True,
                dtype=np.float32,
            ),
        ),
        (
            "char_tfidf",
            TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3, 5),
                max_features=40000,
                min_df=2,
                sublinear_tf=True,
                dtype=np.float32,
            ),
        ),
    ])

## 3. Model Implementation Steps

In [ ]:
def build_transformer(labels_array, num_train_batches):
    counts = np.bincount(labels_array, minlength=num_classes)
    weights = np.power(len(labels_array) / (num_classes * np.maximum(counts, 1)), 0.7)
    weight_tensor = torch.tensor(weights, dtype=torch.float32, device=device)

    net = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=num_classes,
        ignore_mismatched_sizes=True,
        use_safetensors=True
    ).to(device)

    if USE_GRAD_CHECKPOINTING and hasattr(net, "gradient_checkpointing_enable"):
        net.gradient_checkpointing_enable()
        if hasattr(net.config, "use_cache"):
            net.config.use_cache = False

    loss_fn = nn.CrossEntropyLoss(weight=weight_tensor, label_smoothing=LABEL_SMOOTHING)
    opt = torch.optim.AdamW(net.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    opt_steps_per_epoch = max(math.ceil(num_train_batches / GRAD_ACCUM_STEPS), 1)
    total_steps = max(opt_steps_per_epoch * EPOCHS, 1)
    sched = get_cosine_schedule_with_warmup(
        opt,
        num_warmup_steps=int(total_steps * WARMUP_RATIO),
        num_training_steps=total_steps
    )
    scaler = torch.amp.GradScaler("cuda", enabled=use_grad_scaler)
    return net, loss_fn, opt, sched, scaler


def train_epoch_amp(net, loader, loss_fn, opt, sched, scaler):
    net.train()
    opt.zero_grad(set_to_none=True)
    total_loss = 0.0
    steps_done = 0

    for step, batch in enumerate(loader, start=1):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=use_cuda, dtype=amp_dtype):
            outputs = net(input_ids=input_ids, attention_mask=attention_mask)
            raw_loss = loss_fn(outputs.logits, labels)

        if not torch.isfinite(raw_loss):
            opt.zero_grad(set_to_none=True)
            continue

        scaled_loss = raw_loss / GRAD_ACCUM_STEPS
        if use_grad_scaler:
            scaler.scale(scaled_loss).backward()
        else:
            scaled_loss.backward()

        total_loss += raw_loss.item()
        steps_done += 1

        if (step % GRAD_ACCUM_STEPS == 0) or (step == len(loader)):
            if use_grad_scaler:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
                scaler.step(opt)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
                opt.step()
            opt.zero_grad(set_to_none=True)
            sched.step()

    return total_loss / max(steps_done, 1)


def predict_loader_probs(net, loader):
    net.eval()
    all_probs = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=use_cuda, dtype=amp_dtype):
                outputs = net(input_ids=input_ids, attention_mask=attention_mask)
                probs = torch.softmax(outputs.logits.float(), dim=-1)
            all_probs.append(probs.cpu().numpy())
    return np.vstack(all_probs)


def optimize_class_multipliers(probs, y_true, n_restarts=10):
    best_w = np.ones(probs.shape[1], dtype=np.float64)
    best_score = f1_score(y_true, np.argmax(probs, axis=1), average="macro")
    rng = np.random.default_rng(RANDOM_STATE)

    starts = [np.ones(probs.shape[1], dtype=np.float64)]
    for _ in range(n_restarts):
        starts.append(rng.uniform(0.80, 1.30, size=probs.shape[1]))

    grid = np.linspace(0.75, 1.35, 25)
    for w0 in starts:
        w = w0.copy()
        for _ in range(2):
            for c in range(probs.shape[1]):
                c_best = w[c]
                for g in grid:
                    w_try = w.copy()
                    w_try[c] = g
                    sc = f1_score(y_true, np.argmax(probs * w_try, axis=1), average="macro")
                    if sc > best_score:
                        best_score = sc
                        best_w = w_try.copy()
                        c_best = g
                w[c] = c_best

        def obj(vec):
            return -f1_score(y_true, np.argmax(probs * vec, axis=1), average="macro")

        res = minimize(obj, w, method="Powell", options={"maxiter": 80, "ftol": 1e-4})
        if -res.fun > best_score:
            best_score = -res.fun
            best_w = res.x.copy()

    return best_w, best_score



# 5. Stratified 5-Fold Training (RoBERTa-large + TF-IDF)

t0 = time.time()
num_workers = 2 if os.cpu_count() and os.cpu_count() >= 2 else 0
test_ds = PreTokenizedEmotionDataset(df_test["text_clean"], labels=None)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 4, shuffle=False, num_workers=num_workers, pin_memory=use_cuda)

oof_roberta = np.zeros((len(df_train), num_classes), dtype=np.float64)
oof_tfidf = np.zeros((len(df_train), num_classes), dtype=np.float64)
test_roberta_folds = np.zeros((N_SPLITS, len(df_test), num_classes), dtype=np.float64)
test_tfidf_folds = np.zeros((N_SPLITS, len(df_test), num_classes), dtype=np.float64)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
texts_all = df_train["text_clean"].tolist()
texts_test = df_test["text_clean"].tolist()

for fold, (tr_idx, va_idx) in enumerate(skf.split(texts_all, y_all), start=1):
    fold_start = time.time()
    seed_everything(RANDOM_STATE + fold)

    # --- 5a. TF-IDF + LogisticRegression + Calibrated LinearSVC ---
    vec = create_tfidf_vectorizer()
    X_tr_tf = vec.fit_transform([texts_all[i] for i in tr_idx])
    X_va_tf = vec.transform([texts_all[i] for i in va_idx])
    X_te_tf = vec.transform(texts_test)

    lr_clf = LogisticRegression(C=2.0, class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)
    lr_clf.fit(X_tr_tf, y_all[tr_idx])

    svc_clf = CalibratedClassifierCV(
        estimator=LinearSVC(C=0.1, class_weight="balanced", random_state=RANDOM_STATE),
        cv=3
    )
    svc_clf.fit(X_tr_tf, y_all[tr_idx])

    p_va_tf = 0.5 * lr_clf.predict_proba(X_va_tf) + 0.5 * svc_clf.predict_proba(X_va_tf)
    p_te_tf = 0.5 * lr_clf.predict_proba(X_te_tf) + 0.5 * svc_clf.predict_proba(X_te_tf)
    oof_tfidf[va_idx] = p_va_tf
    test_tfidf_folds[fold - 1] = p_te_tf

    # --- 5b. RoBERTa-large fine-tuning ---
    tr_ds = PreTokenizedEmotionDataset([texts_all[i] for i in tr_idx], y_all[tr_idx])
    va_ds = PreTokenizedEmotionDataset([texts_all[i] for i in va_idx], y_all[va_idx])

    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers, pin_memory=use_cuda)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE * 4, shuffle=False, num_workers=num_workers, pin_memory=use_cuda)

    model, criterion, optimizer, scheduler, scaler = build_transformer(y_all[tr_idx], len(tr_loader))

    best_fold_f1 = -1.0
    best_fold_va_probs = None
    best_fold_state = None

    for epoch in range(1, EPOCHS + 1):
        ep_loss = train_epoch_amp(model, tr_loader, criterion, optimizer, scheduler, scaler)
        va_probs = predict_loader_probs(model, va_loader)
        va_f1 = f1_score(y_all[va_idx], np.argmax(va_probs, axis=1), average="macro")
        print(f"[Fold {fold}/{N_SPLITS}] Epoch {epoch}/{EPOCHS} | Loss: {ep_loss:.4f} | Val Macro F1: {va_f1:.4f}")

        if va_f1 > best_fold_f1:
            best_fold_f1 = va_f1
            best_fold_va_probs = va_probs
            best_fold_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    oof_roberta[va_idx] = best_fold_va_probs
    if best_fold_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_fold_state.items()})
    test_roberta_folds[fold - 1] = predict_loader_probs(model, test_loader)

    tf_f1 = f1_score(y_all[va_idx], np.argmax(p_va_tf, axis=1), average="macro")
    print(f"--> Fold {fold} done in {time.time() - fold_start:.1f}s | TF-IDF F1: {tf_f1:.4f} | Best RoBERTa F1: {best_fold_f1:.4f}\n")

    clear_cuda_memory(model, criterion, optimizer, scheduler, scaler, tr_loader, va_loader, tr_ds, va_ds, best_fold_state)


# 6. OOF Evaluation, Blending & Multiplier Tuning

test_roberta_mean = test_roberta_folds.mean(axis=0)
test_tfidf_mean = test_tfidf_folds.mean(axis=0)

oof_rob_f1 = f1_score(y_all, np.argmax(oof_roberta, axis=1), average="macro")
oof_tf_f1 = f1_score(y_all, np.argmax(oof_tfidf, axis=1), average="macro")

best_alpha = 1.0
best_blend_f1 = oof_rob_f1
for alpha in np.linspace(0.60, 1.00, 41):
    oof_blend = alpha * oof_roberta + (1.0 - alpha) * oof_tfidf
    sc = f1_score(y_all, np.argmax(oof_blend, axis=1), average="macro")
    if sc > best_blend_f1:
        best_blend_f1 = sc
        best_alpha = float(alpha)

oof_best_probs = best_alpha * oof_roberta + (1.0 - best_alpha) * oof_tfidf
test_best_probs = best_alpha * test_roberta_mean + (1.0 - best_alpha) * test_tfidf_mean

print(f"OOF Macro F1 (TF-IDF only):        {oof_tf_f1:.4f}")
print(f"OOF Macro F1 (RoBERTa-large only): {oof_rob_f1:.4f}")
print(f"OOF Macro F1 (Blend alpha={best_alpha:.2f}): {best_blend_f1:.4f}")

tuned_w, tuned_oof_f1 = optimize_class_multipliers(oof_best_probs, y_all)
use_tuned_w = (tuned_oof_f1 - best_blend_f1) >= 0.002
final_w = tuned_w if use_tuned_w else np.ones(num_classes, dtype=np.float64)

final_oof_preds = np.argmax(oof_best_probs * final_w, axis=1)
final_oof_f1 = f1_score(y_all, final_oof_preds, average="macro")
final_oof_acc = accuracy_score(y_all, final_oof_preds)

print(f"\n>>> Final OOF Macro F1: {final_oof_f1:.4f} | Final OOF Acc: {final_oof_acc:.4f}")
if use_tuned_w:
    print("Applied class multipliers:", np.round(final_w, 3))
print("\n" + classification_report(y_all, final_oof_preds, target_names=label_encoder.classes_, digits=4))



# 7. Save Submission

test_pred_id = np.argmax(test_best_probs * final_w, axis=1)
test_pred_label = label_encoder.inverse_transform(test_pred_id)

submission = pd.DataFrame({
    "id": df_test["post_id"],
    "label": test_pred_label,
})
output_path = "sampleSubmission.csv"
submission.to_csv(output_path, index=False)

print(f"\nTotal runtime: {time.time() - t0:.1f}s")
print("Submission file saved:", output_path)
print("Number of test predictions:", len(submission))
print("\nTrain label distribution (normalized):")
print(df_train["label"].value_counts(normalize=True).round(4))
print("\nPredicted test label distribution:")
print(submission["label"].value_counts())
display(submission.head(10))